# SPC3 Continuous Acquisition — Logic Layer Test

Test continuous acquisition through the qudi `CameraLogic` layer.  
Requires qudi to be running with `SPUD202603.cfg` loaded.

## Important: reload modules after logic edits

If you changed files under `qudi-iqo-modules/src/qudi/logic` or `qudi-iqo-modules/src/qudi/hardware`, restart qudi before running this notebook again.

Otherwise the notebook kernel may still call stale module code and show old tracebacks/line numbers.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime


# Qudi exposes module instances as globals in the notebook kernel.
# camera_logic is the CameraLogic module, camera_SPC3 is the hardware module.
logic = camera_logic
hw = camera_SPC3

## Verify module state

In [2]:
# Check that modules are active and camera is ready
print(f'Logic module state : {logic.module_state()}')
print(f'Camera name        : {hw.get_name()}')
print(f'Camera size        : {hw.get_size()}')
print(f'Exposure           : {logic.get_exposure() * 1e6:.2f} us')
print(f'Trigger mode       : {hw.get_trigger_mode()}')
print(f'Gate mode          : {hw.get_gate_mode()}')
print(f'Camera ready       : {hw.get_ready_state()}')

Logic module state : idle
Camera name        : SPC3 SPC3-B sn00032
Camera size        : (32, 32)
Exposure           : 20.80 us
Trigger mode       : multiple_trigger
Gate mode          : coarse
Camera ready       : True


## Start continuous acquisition

In [8]:

data_dir = r'C:\Users\SPUD1\Documents\experiment_workspace\SPAD data\spc3\20260315'
fname = 'contacq_ODMR_ensemble'
filepath = os.path.join(data_dir, fname)

# Start continuous acquisition via the logic layer.
ok = logic.start_continuous_acquisition(filepath)
print(f'Acquisition started: {ok}  |  {datetime.now():%Y-%m-%d %H:%M:%S}')

Acquisition started: True  |  2026-03-15 16:46:48


## Monitor progress
Run this cell repeatedly to check acquisition status.

In [8]:
status = logic.get_continuous_status()
phase = status.get('phase', 'unknown')
last_error = status.get('last_error')

if status['active']:
    print(f"Running: {status['elapsed_s']:.1f} s, "
          f"{status['bytes'] / 1e6:.2f} MB written, "
          f"{status['errors']} transient errors, "
          f"phase={phase}")
    if last_error:
        print(f"Last SDK error: {last_error}")
else:
    print(f"No continuous acquisition running (phase={phase})")
    if last_error:
        print(f"Last SDK error: {last_error}")

EOFError: stream has been closed

## Stop continuous acquisition

In [7]:
try:
    result = logic.stop_continuous_acquisition()
except Exception as exc:
    result = None
    print(f"Stop call raised exception: {exc}")

if result:
    print(f"Stopped: {result['elapsed_s']:.2f} s, "
          f"{result['bytes'] / 1e6:.2f} MB, "
          f"{result['errors']} errors")
    print(f"File: {result['filename']}.spc3")
    print(f"Stop phase: {result.get('phase', 'unknown')}  |  stop_ok={result.get('stop_ok', True)}")
    if result.get('drained_bytes') is not None:
        print(f"Final drain before stop: {result['drained_bytes'] / 1e6:.2f} MB")
    if result.get('stop_error'):
        print(f"Stop error: {result['stop_error']}")
    if result['errors'] > 0:
        print(f"\n⚠ {result['errors']} transient SDK errors occurred during acquisition.")
        print("  Check qudi log for timestamped diagnostics.")
else:
    print("No acquisition was running or stop failed before summary return")

Stopped: 12.61 s, 0.00 MB, 0 errors
File: C:\Users\SPUD1\Documents\experiment_workspace\SPAD data\spc3\20260315\contacq_ODMR_ensemble_shift1.spc3
Stop phase: stopped  |  stop_ok=True
Final drain before stop: 0.00 MB


## Load and display acquired data

In [ ]:
from qudi.hardware.camera.SPC3.spc import SPC3

spc3_file = filepath + '.spc3'
frames, header = SPC3.ReadSPC3DataFile(spc3_file)
n_counters, n_frames, n_rows, n_cols = frames.shape
print(f'Loaded: {n_frames} frames, {n_rows}×{n_cols} pixels, {n_counters} counter(s)')
print(f'dtype: {frames.dtype}, range: [{frames.min()}, {frames.max()}]')

In [ ]:
# Display first, middle, and last frames
indices = [0, n_frames // 2, n_frames - 1]
fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 4))
for ax, idx in zip(axes, indices):
    im = ax.imshow(frames[0, idx], cmap='inferno', origin='lower', interpolation='none')
    ax.set_title(f'Frame {idx}')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Plot total counts per frame over time to see signal evolution
totals = frames[0].sum(axis=(1, 2))  # sum each frame spatially
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(totals, lw=0.5)
ax.set_xlabel('Frame index')
ax.set_ylabel('Total counts')
ax.set_title('Per-frame total counts')
plt.tight_layout()
plt.show()